In [9]:
import torch
import torch.nn as nn

print(torch.__version__)

2.8.0


In [10]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from seq2seq.model.encoder import Encoder

In [14]:
V_SRC_SIZE = 100
V_TGT_SIZE = 100

EMBEDDING_DIM = 32
HIDDEN_DIM = 64
LAYERS = 2

BATCH_SIZE = 4
SOURCE_LENGTH = 7

In [16]:
class Decoder(nn.Module):
    def __init__(
        self,
        v_tgt_size,
        embedding_dim,
        hidden_dim,
        layers,
    ) -> None:
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=v_tgt_size,
            embedding_dim=embedding_dim,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=layers,
            batch_first=True,
        )

        self.output = nn.Linear(
            in_features=hidden_dim,
            out_features=v_tgt_size,
        )

    def forward(
        self,
        input_token,
        hidden,
        cell
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        embedded = self.embedding(input_token).unsqueeze(1)
        outputs, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell),
        )
        outputs = outputs.squeeze(1)
        logits = self.output(outputs)

        return logits, hidden, cell

In [25]:
test_encoder = Encoder(
    V_SRC_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    LAYERS
)

test_decoder = Decoder(
    V_TGT_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    LAYERS,
)

In [26]:
src = torch.randint(
    low=0,
    high=V_SRC_SIZE,
    size=(BATCH_SIZE, SOURCE_LENGTH)
)

In [27]:
encoded_outputs, encoded_hidden, encoded_cell = test_encoder(src)

encoded_outputs.shape, encoded_hidden.shape, encoded_cell.shape

(torch.Size([4, 7, 64]), torch.Size([2, 4, 64]), torch.Size([2, 4, 64]))

In [33]:
SOS_IDX = 1

input_token = torch.full(
    size=(BATCH_SIZE,),
    fill_value=SOS_IDX,
    dtype=torch.long,
)

In [34]:
logits, hidden, cell = test_decoder(input_token, encoded_hidden, encoded_cell)

logits.shape, hidden.shape, cell.shape

(torch.Size([4, 100]), torch.Size([2, 4, 64]), torch.Size([2, 4, 64]))

In [32]:
predicted_token = logits.argmax(dim=1)

predicted_token

tensor([ 4,  4,  4, 36])